# Methanol Production from Natural Gas

This notebook simulates a methanol production process from natural gas using NeqSim.
The process consists of two main stages:

1. **Steam Methane Reforming (SMR)** — Convert CH₄ + H₂O to syngas (CO + H₂)
2. **Methanol Synthesis** — Convert CO + 2H₂ → CH₃OH

## Key Reactions

### Steam Methane Reforming (endothermic, 800-900°C)
$$\text{CH}_4 + \text{H}_2\text{O} \rightleftharpoons \text{CO} + 3\text{H}_2 \quad \Delta H = +206 \text{ kJ/mol}$$

### Water-Gas Shift (mildly exothermic)
$$\text{CO} + \text{H}_2\text{O} \rightleftharpoons \text{CO}_2 + \text{H}_2 \quad \Delta H = -41 \text{ kJ/mol}$$

### Methanol Synthesis (exothermic, 200-300°C, 50-100 bar)
$$\text{CO} + 2\text{H}_2 \rightleftharpoons \text{CH}_3\text{OH} \quad \Delta H = -91 \text{ kJ/mol}$$
$$\text{CO}_2 + 3\text{H}_2 \rightleftharpoons \text{CH}_3\text{OH} + \text{H}_2\text{O} \quad \Delta H = -49 \text{ kJ/mol}$$

We use the NeqSim **GibbsReactor** for equilibrium calculations of both stages.

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
Separator = jneqsim.process.equipment.separator.Separator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
GibbsReactor = jneqsim.process.equipment.reactor.GibbsReactor
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

print('NeqSim loaded successfully')

## 1. Steam Methane Reforming (SMR) Equilibrium

Study the equilibrium of steam reforming at different temperatures.
The feed is methane + steam with a steam-to-carbon ratio (S/C) of 3.

In [ ]:
# SMR equilibrium vs temperature
temps_smr = np.arange(400, 1050, 25)
P_smr = 25.0  # bara (typical SMR pressure)
sc_ratio = 3.0  # Steam-to-Carbon ratio

ch4_conv = []
h2_frac = []
co_frac = []
co2_frac = []

for t in temps_smr:
    try:
        feed_smr = SystemSrkEos(273.15 + float(t), P_smr)
        feed_smr.addComponent('methane', 1.0)  # 1 mol CH4
        feed_smr.addComponent('water', sc_ratio)  # 3 mol H2O
        feed_smr.addComponent('hydrogen', 1e-10)
        feed_smr.addComponent('CO', 1e-10)
        feed_smr.addComponent('CO2', 1e-10)
        feed_smr.setMixingRule('classic')

        s = Stream('SMR Feed', feed_smr)
        s.setFlowRate(10000.0, 'kg/hr')
        s.setTemperature(float(t), 'C')
        s.setPressure(P_smr, 'bara')

        p = ProcessSystem()
        p.add(s)

        rx = GibbsReactor('SMR', s)
        rx.setEnergyMode('isothermal')
        rx.setMaxIterations(5000)
        p.add(rx)

        p.run()

        out = rx.getOutletStream().getFluid()
        x_ch4 = float(out.getPhase(0).getComponent('methane').getx())
        x_h2 = float(out.getPhase(0).getComponent('hydrogen').getx())
        x_co = float(out.getPhase(0).getComponent('CO').getx())
        x_co2 = float(out.getPhase(0).getComponent('CO2').getx())

        # CH4 conversion based on mole fraction change
        # Initial: 1/(1+3) = 0.25 mol fraction
        # The moles increase as reaction proceeds
        ch4_conv.append((1.0 - x_ch4/0.25) * 100 if x_ch4 < 0.25 else 0)
        h2_frac.append(x_h2 * 100)
        co_frac.append(x_co * 100)
        co2_frac.append(x_co2 * 100)
    except Exception:
        ch4_conv.append(float('nan'))
        h2_frac.append(float('nan'))
        co_frac.append(float('nan'))
        co2_frac.append(float('nan'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(temps_smr, h2_frac, 'g-o', linewidth=2, markersize=4, label='H₂')
ax1.plot(temps_smr, co_frac, 'b-s', linewidth=2, markersize=4, label='CO')
ax1.plot(temps_smr, co2_frac, 'r-^', linewidth=2, markersize=4, label='CO₂')
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('Mole Fraction (%)', fontsize=12)
ax1.set_title(f'SMR Syngas Composition at {P_smr} bara, S/C={sc_ratio}')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Effect of S/C ratio at 850°C
sc_ratios = np.arange(1.0, 6.0, 0.5)
h2_co_ratios = []

for sc in sc_ratios:
    try:
        f = SystemSrkEos(273.15 + 850.0, P_smr)
        f.addComponent('methane', 1.0)
        f.addComponent('water', float(sc))
        f.addComponent('hydrogen', 1e-10)
        f.addComponent('CO', 1e-10)
        f.addComponent('CO2', 1e-10)
        f.setMixingRule('classic')

        s2 = Stream('Feed', f)
        s2.setFlowRate(10000.0, 'kg/hr')
        s2.setTemperature(850.0, 'C')
        s2.setPressure(P_smr, 'bara')

        p2 = ProcessSystem()
        p2.add(s2)
        r2 = GibbsReactor('SMR', s2)
        r2.setEnergyMode('isothermal')
        r2.setMaxIterations(5000)
        p2.add(r2)
        p2.run()

        out2 = r2.getOutletStream().getFluid()
        x_h2_ = float(out2.getPhase(0).getComponent('hydrogen').getx())
        x_co_ = float(out2.getPhase(0).getComponent('CO').getx())
        h2_co_ratios.append(x_h2_ / x_co_ if x_co_ > 1e-6 else float('nan'))
    except Exception:
        h2_co_ratios.append(float('nan'))

ax2.plot(sc_ratios, h2_co_ratios, 'b-o', linewidth=2, markersize=6)
ax2.axhline(y=2.0, color='r', linestyle='--', alpha=0.5, label='Stoichiometric (H₂/CO=2)')
ax2.set_xlabel('Steam-to-Carbon Ratio', fontsize=12)
ax2.set_ylabel('H₂/CO Molar Ratio', fontsize=12)
ax2.set_title('Syngas H₂/CO Ratio at 850°C, 25 bara')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('smr_equilibrium.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Methanol Synthesis Equilibrium

Study the equilibrium at different temperatures for the methanol synthesis step.
The feed is syngas (H₂ + CO) with stoichiometric ratio H₂/CO = 2.

In [ ]:
# Methanol synthesis equilibrium at different temperatures
temps_meoh = np.arange(150, 400, 10)
P_meoh = 80.0  # bara (typical synthesis pressure)

meoh_yield = []

for t in temps_meoh:
    try:
        syngas = SystemSrkEos(273.15 + float(t), P_meoh)
        syngas.addComponent('hydrogen', 2.0)  # H2/CO = 2
        syngas.addComponent('CO', 1.0)
        syngas.addComponent('CO2', 0.05)  # small CO2
        syngas.addComponent('methanol', 1e-10)
        syngas.addComponent('water', 1e-10)
        syngas.addComponent('methane', 0.05)  # inert CH4
        syngas.setMixingRule('classic')

        s = Stream('Syngas', syngas)
        s.setFlowRate(10000.0, 'kg/hr')
        s.setTemperature(float(t), 'C')
        s.setPressure(P_meoh, 'bara')

        proc = ProcessSystem()
        proc.add(s)

        rx = GibbsReactor('MeOH Reactor', s)
        rx.setEnergyMode('isothermal')
        rx.setMaxIterations(5000)
        rx.setComponentAsInert('methane')  # CH4 is inert
        proc.add(rx)

        proc.run()

        out = rx.getOutletStream().getFluid()
        x_meoh = float(out.getPhase(0).getComponent('methanol').getx())
        meoh_yield.append(x_meoh * 100)
    except Exception:
        meoh_yield.append(float('nan'))

# Pressure effect at 250°C
pressures_meoh = np.arange(20, 120, 5)
meoh_yield_P = []

for p in pressures_meoh:
    try:
        sg = SystemSrkEos(273.15 + 250.0, float(p))
        sg.addComponent('hydrogen', 2.0)
        sg.addComponent('CO', 1.0)
        sg.addComponent('CO2', 0.05)
        sg.addComponent('methanol', 1e-10)
        sg.addComponent('water', 1e-10)
        sg.addComponent('methane', 0.05)
        sg.setMixingRule('classic')

        s2 = Stream('SG', sg)
        s2.setFlowRate(10000.0, 'kg/hr')
        s2.setTemperature(250.0, 'C')
        s2.setPressure(float(p), 'bara')

        p2 = ProcessSystem()
        p2.add(s2)
        r2 = GibbsReactor('MeOH', s2)
        r2.setEnergyMode('isothermal')
        r2.setMaxIterations(5000)
        r2.setComponentAsInert('methane')
        p2.add(r2)
        p2.run()

        x_m = float(r2.getOutletStream().getFluid().getPhase(0).getComponent('methanol').getx())
        meoh_yield_P.append(x_m * 100)
    except Exception:
        meoh_yield_P.append(float('nan'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(temps_meoh, meoh_yield, 'g-o', linewidth=2, markersize=4)
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('CH₃OH Mole Fraction (%)', fontsize=12)
ax1.set_title(f'MeOH Synthesis Equilibrium at {P_meoh} bara')
ax1.axvspan(200, 300, alpha=0.1, color='green', label='Typical operating range')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(pressures_meoh, meoh_yield_P, 'b-o', linewidth=2, markersize=4)
ax2.set_xlabel('Pressure (bara)', fontsize=12)
ax2.set_ylabel('CH₃OH Mole Fraction (%)', fontsize=12)
ax2.set_title('MeOH Synthesis: Pressure Effect at 250°C')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('meoh_synthesis_equilibrium.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Complete Methanol Production Process

Chain the SMR reactor, heat recovery, compression, and methanol synthesis reactor.

In [ ]:
# Complete methanol process
# Step 1: SMR reactor feed (CH4 + steam)
smr_fluid = SystemSrkEos(273.15 + 850.0, 25.0)
smr_fluid.addComponent('methane', 1.0)
smr_fluid.addComponent('water', 3.0)  # S/C = 3
smr_fluid.addComponent('hydrogen', 1e-10)
smr_fluid.addComponent('CO', 1e-10)
smr_fluid.addComponent('CO2', 1e-10)
smr_fluid.addComponent('methanol', 1e-10)
smr_fluid.setMixingRule('classic')

smr_feed = Stream('SMR Feed', smr_fluid)
smr_feed.setFlowRate(20000.0, 'kg/hr')
smr_feed.setTemperature(850.0, 'C')
smr_feed.setPressure(25.0, 'bara')

meoh_process = ProcessSystem()
meoh_process.add(smr_feed)

# SMR Reactor
smr_reactor = GibbsReactor('SMR Reactor', smr_feed)
smr_reactor.setEnergyMode('isothermal')
smr_reactor.setMaxIterations(5000)
meoh_process.add(smr_reactor)

# Cool syngas (heat recovery)
syngas_cooler = Cooler('Syngas Cooler', smr_reactor.getOutletStream())
syngas_cooler.setOutTemperature(273.15 + 40.0)
meoh_process.add(syngas_cooler)

# Separate condensed water
water_sep = Separator('Water KO Drum', syngas_cooler.getOutletStream())
meoh_process.add(water_sep)

# Compress syngas to synthesis pressure
syngas_comp = Compressor('Syngas Compressor', water_sep.getGasOutStream())
syngas_comp.setOutletPressure(80.0)
syngas_comp.setIsentropicEfficiency(0.78)
meoh_process.add(syngas_comp)

# Cool compressed syngas
syngas_ac = Cooler('Syngas AC', syngas_comp.getOutletStream())
syngas_ac.setOutTemperature(273.15 + 250.0)
meoh_process.add(syngas_ac)

# Methanol Synthesis Reactor
meoh_reactor = GibbsReactor('MeOH Reactor', syngas_ac.getOutletStream())
meoh_reactor.setEnergyMode('isothermal')
meoh_reactor.setMaxIterations(5000)
meoh_process.add(meoh_reactor)

# Cool reactor product
product_cooler = Cooler('MeOH Product Cooler', meoh_reactor.getOutletStream())
product_cooler.setOutTemperature(273.15 + 40.0)
meoh_process.add(product_cooler)

# Separate crude methanol from unreacted gas
meoh_sep = Separator('MeOH Separator', product_cooler.getOutletStream())
meoh_process.add(meoh_sep)

meoh_process.run()

print('=== Complete Methanol Production Process ===')
print(f'Feed Rate:        {smr_feed.getFlowRate("kg/hr"):.0f} kg/hr')
print()

# SMR outlet composition
smr_out = smr_reactor.getOutletStream().getFluid()
print('SMR Reactor Outlet (syngas):')
for i in range(int(smr_out.getNumberOfComponents())):
    comp = smr_out.getPhase(0).getComponent(i)
    x = float(comp.getx()) * 100
    if x > 0.01:
        print(f'  {str(comp.getComponentName()):12s}: {x:7.2f} mol%')

# MeOH reactor outlet
meoh_out = meoh_reactor.getOutletStream().getFluid()
print('\nMeOH Reactor Outlet:')
for i in range(int(meoh_out.getNumberOfComponents())):
    comp = meoh_out.getPhase(0).getComponent(i)
    x = float(comp.getx()) * 100
    if x > 0.01:
        print(f'  {str(comp.getComponentName()):12s}: {x:7.2f} mol%')

# Power consumption
comp_power = syngas_comp.getPower('kW')
print(f'\nSyngas Compressor Power: {comp_power:.0f} kW')
print(f'Crude MeOH liquid rate:  {meoh_sep.getLiquidOutStream().getFlowRate("kg/hr"):.0f} kg/hr')
print(f'Off-gas rate:            {meoh_sep.getGasOutStream().getFlowRate("kg/hr"):.0f} kg/hr')

## Summary

Key findings:

- **SMR**: Best equilibrium at high temperature (>800°C), low pressure, S/C ratio ~3
- **MeOH Synthesis**: Favored by low temperature (<300°C) and high pressure (50-100 bar)
- The opposing temperature requirements necessitate two separate reactors
- Per-pass methanol conversion is typically 25-40% due to equilibrium limitations
- Industrial processes use recycle loops to achieve overall conversion >95%

### Process Optimization Opportunities
- Adjust S/C ratio for optimal H₂/CO ratio (stoichiometric = 2)
- Multi-stage synthesis with intermediate methanol removal
- Autothermal reforming (ATR) instead of SMR for better heat integration
- CO₂ co-feed to adjust syngas composition